In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used
# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [1.1732, 1.3325, 1.1732+1.3325]

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [2]:
files = sorted((project_root/'data/260323').glob('SUBT_*'))
acqs = [SiphraAcquisition(file) for file in files]

In [3]:
acqs

[SIPHRAAcquisition(File: 'SUBT_01_EnergyResolution_thr30_gain1over100_1over3_Background'),
 SIPHRAAcquisition(File: 'SUBT_02_EnergyResolution_thr30_gain1over100_1over3_Cs137'),
 SIPHRAAcquisition(File: 'SUBT_03_EnergyResolution_thr30_gain1over100_1over3_Background2'),
 SIPHRAAcquisition(File: 'SUBT_04_EnergyResolution_thr30_gain1over100_1over3_Na22'),
 SIPHRAAcquisition(File: 'SUBT_05_EnergyResolution_thr30_gain1over100_1over3_Background4'),
 SIPHRAAcquisition(File: 'SUBT_06_EnergyResolution_thr30_gain1over100_1over3_Co60'),
 SIPHRAAcquisition(File: 'SUBT_07_EnergyResolution_thr30_gain1over100_1over3_Background6'),
 SIPHRAAcquisition(File: 'SUBT_08_EnergyResolution_thr30_gain1over100_1over3_Am241'),
 SIPHRAAcquisition(File: 'SUBT_09_EnergyResolution_thr30_gain1over100_1over3_Background8')]

In [4]:
# histograms
hists = {}
sources = []
for sgnl, bg in zip(acqs[1::2], acqs[2::2]):
    filepath = sgnl.filepath.stem
    src = (MetadataLoader.load(sgnl.metadataFile)).source
    sources.append(src)
    print(src)
    # Signal and Background
    hist_sgnl = ROOT.TH1F(f"{src} Signal", "", BITS12, 0, BITS12)
    hist_bg = ROOT.TH1F(f"{src} Background", "", BITS12, 0 , BITS12)
    hist_sgnl.Fill(sgnl['s']/len(sgnl.active_chs))
    hist_bg.Fill(bg['s']/len(bg.active_chs))
    hist_bg.Scale(sgnl.exposure/bg.exposure)
    # Clean spectrum
    hist_clean = hist_sgnl.Clone(f"{src} Clean")
    hist_clean.Add(hist_bg, -1)
    for hist in [hist_sgnl, hist_bg, hist_clean]:
        # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
        hist.GetXaxis().SetTitle("ADC channel number")
        hist.GetYaxis().SetTitle(r"Normalized counts")
    hists[src] = hist_sgnl
    hists[f"{src}_BG"] = hist_bg
    hists[f"{src}_clean"] = hist_clean
    del hist_sgnl, hist_bg
print(hists)

Cs-137
Na-22
Co-60
Am-241
{'Cs-137': <cppyy.gbl.TH1F object at 0x61d0d8f0a860>, 'Cs-137_BG': <cppyy.gbl.TH1F object at 0x61d0d3124d80>, 'Cs-137_clean': <cppyy.gbl.TH1F object at 0x61d0d3e17050>, 'Na-22': <cppyy.gbl.TH1F object at 0x61d0d94204c0>, 'Na-22_BG': <cppyy.gbl.TH1F object at 0x61d0d92e7a20>, 'Na-22_clean': <cppyy.gbl.TH1F object at 0x61d0d93aed20>, 'Co-60': <cppyy.gbl.TH1F object at 0x61d0d9145ac0>, 'Co-60_BG': <cppyy.gbl.TH1F object at 0x61d0d3d83810>, 'Co-60_clean': <cppyy.gbl.TH1F object at 0x61d0d86e60d0>, 'Am-241': <cppyy.gbl.TH1F object at 0x61d0d837cc90>, 'Am-241_BG': <cppyy.gbl.TH1F object at 0x61d0d94c3d10>, 'Am-241_clean': <cppyy.gbl.TH1F object at 0x61d0d94ec620>}


In [5]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 1600, 1200)
cv.Divide(2,2)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(4)]

for idx, (src, lg) in enumerate(zip(sources, lgds)):   
    cv.cd(idx+1)
    
    sgnl = hists[src]
    bg = hists[src+'_BG']
    clean = hists[src+'_clean']
    
    lg.AddEntry(sgnl, "Signal")
    lg.AddEntry(bg, "Background")
    lg.AddEntry(clean, "Subtracted")
    sgnl.SetLineColor(colors[0])
    bg.SetLineColor(colors[1])
    clean.SetLineColor(colors[2])
    sgnl.SetTitle(src)
    sgnl.Draw("hist")
    bg.Draw("sames hist")
    clean.Draw("sames hist")
    ROOT.gPad.Update()
    for i, sp in enumerate([sgnl, bg, clean]):
        st = sp.FindObject("stats")
        # print(type(st))
        st.SetY1NDC(Yinit - i*0.08)
        st.SetY2NDC(0.1)
    lg.Draw()
    cv.cd(idx+1).SetLogy()
    cv.Draw()




In [6]:
fit_xrange = {
    'Cs-137': [(100, 270)],
    'Na-22': [(325, 408), (436,508)],
    'Co-60': [(200, 361), (419, 520)],
    'Am-241': [(8, 30)]
}

# Fit Cs-137


In [18]:
acqs[2::2]

[SIPHRAAcquisition(File: 'SUBT_03_EnergyResolution_thr30_gain1over100_1over3_Background2'),
 SIPHRAAcquisition(File: 'SUBT_05_EnergyResolution_thr30_gain1over100_1over3_Background4'),
 SIPHRAAcquisition(File: 'SUBT_07_EnergyResolution_thr30_gain1over100_1over3_Background6'),
 SIPHRAAcquisition(File: 'SUBT_09_EnergyResolution_thr30_gain1over100_1over3_Background8')]